# fMRI mixed-site model sweep

This experiment tests whether non-LOSO subject-level evaluation can improve the fMRI result without subject leakage. It reports repeated mixed-site CV and a pre-specified 80/20 holdout comparable to the earlier structural Swin-T experiment. Site/confound ablations are mandatory so a higher score is not misattributed to neuroimaging.

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_MOUNT>')
from pathlib import Path
import warnings, numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
warnings.filterwarnings('ignore')
ROOT=Path('<DATA_DIR>'); BENCH=ROOT/'fmri'/'strict_loso_benchmark'
BRAINLM=ROOT/'fmri'/'brainlm_a424'; OUT=ROOT/'fmri'/'mixed_site_model_sweep'; OUT.mkdir(parents=True,exist_ok=True)
cohort=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str})
bank=np.load(BENCH/'a424_feature_bank.npz',allow_pickle=True)
meta=pd.read_csv(BRAINLM/'brainlm_subject_embedding_metadata.csv',dtype={'subject_id':str}); emb=np.load(BRAINLM/'brainlm_subject_embeddings.npy')
emap={s:i for i,s in enumerate(meta.subject_id.astype(str))}; X_blm=np.stack([emb[emap[s]] for s in cohort.subject_id.astype(str)]).astype('float32')
X_fc=bank['fc_summary'].astype('float32'); X_spec=bank['spectral'].astype('float32')
X_conf=cohort[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy('float32')
X_site=pd.get_dummies(cohort.site.astype(str),prefix='site',dtype=float).to_numpy('float32')
y=cohort.label.to_numpy(int); site=cohort.site.astype(str).to_numpy(); strata=np.char.add(np.char.add(site,'__'),y.astype(str))
features={'confounds':X_conf,'site_confounds':np.c_[X_site,X_conf],'fmri_fc_brainlm':np.c_[X_fc,X_blm],
          'fmri_all':np.c_[X_fc,X_spec,X_blm],
          'fmri_all_confounds':np.c_[X_fc,X_spec,X_blm,X_conf],
          'fmri_all_site_confounds':np.c_[X_fc,X_spec,X_blm,X_site,X_conf]}
print(len(y),'subjects;',len(np.unique(site)),'sites', {k:v.shape[1] for k,v in features.items()})

## Repeated subject-level mixed-site CV
Four folds × three repeats. All transformations are fitted inside the training fold. Model settings are fixed before evaluation; no test-fold score is used for selection.

In [ ]:
def build(kind):
    if kind=='logreg': return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(C=.03,class_weight='balanced',max_iter=4000,solver='liblinear'))
    if kind=='rbf_svm': return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),SVC(C=.3,gamma='scale',class_weight='balanced',probability=True,random_state=42))
    if kind=='mlp': return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),MLPClassifier(hidden_layer_sizes=(64,16),alpha=.05,learning_rate_init=3e-4,max_iter=500,early_stopping=True,validation_fraction=.15,n_iter_no_change=25,random_state=42))
configs=[('confounds','logreg'),('site_confounds','logreg'),('fmri_fc_brainlm','logreg'),('fmri_all','logreg'),
         ('fmri_all_confounds','logreg'),('fmri_all_site_confounds','logreg'),
         ('fmri_all','rbf_svm'),('fmri_all_confounds','rbf_svm'),('fmri_all_site_confounds','rbf_svm'),
         ('fmri_all_confounds','mlp'),('fmri_all_site_confounds','mlp')]
outer=RepeatedStratifiedKFold(n_splits=4,n_repeats=3,random_state=2026); rows=[]
for feat,kind in configs:
    X=features[feat]; print('CV',feat,kind,flush=True)
    for fold,(tr,te) in enumerate(outer.split(X,strata),1):
        m=build(kind).fit(X[tr],y[tr]); p=m.predict_proba(X[te])[:,1]; pred=(p>=.5).astype(int)
        rows.append({'feature_set':feat,'model':kind,'fold':fold,'auc':roc_auc_score(y[te],p),
                     'ap':average_precision_score(y[te],p),'balanced_accuracy':balanced_accuracy_score(y[te],pred)})
cv=pd.DataFrame(rows); summary=(cv.groupby(['feature_set','model']).agg(mean_auc=('auc','mean'),sd_auc=('auc','std'),mean_ap=('ap','mean'),mean_balanced_accuracy=('balanced_accuracy','mean')).reset_index().sort_values('mean_auc',ascending=False))
cv.to_csv(OUT/'mixed_site_cv_folds.csv',index=False); summary.to_csv(OUT/'mixed_site_cv_summary.csv',index=False); display(summary.round(3))

## Pre-specified 80/20 comparison split
The split is stratified jointly by site and label with random state 42. All configurations are reported; the best test score is exploratory and is not a replacement for repeated CV.

In [ ]:
split=StratifiedShuffleSplit(n_splits=1,test_size=.2,random_state=42); tr,te=next(split.split(np.zeros(len(y)),strata)); hold=[]
for feat,kind in configs:
    X=features[feat]; m=build(kind).fit(X[tr],y[tr]); p=m.predict_proba(X[te])[:,1]; pred=(p>=.5).astype(int)
    hold.append({'feature_set':feat,'model':kind,'n_train':len(tr),'n_test':len(te),'auc':roc_auc_score(y[te],p),'ap':average_precision_score(y[te],p),'balanced_accuracy':balanced_accuracy_score(y[te],pred)})
hold=pd.DataFrame(hold).sort_values('auc',ascending=False); hold.to_csv(OUT/'mixed_site_holdout_summary.csv',index=False); display(hold.round(3))
print('Target reference: structural Swin-T single-split AUC = 0.774')

## Interpretation rule
A result at or above 0.774 is considered image-derived only if an fMRI-only configuration reaches it and the gain is stable in repeated CV. If only the site/confound-inclusive model reaches it, the score demonstrates mixed-site prediction but not an fMRI biomarker.